In [0]:
# ================= IMPORT CONFIG PROJECT =================

In [0]:
%run ../utils/config_project

In [0]:
# ==================== INSTALL LIBS ====================

%pip install openpyxl

# ==================== IMPORT LIBS ====================
from datetime   import datetime
from pathlib    import Path

import pandas   as pd

import requests
import json
import os 

In [0]:
# ==================== READ CONFIG INI ====================
config = config_ini()

# ==================== DEFINE VARIABLES ====================

#Paths
path_project        = Path(config.get('config','project_path'))
path_landing_zone   = path_project / Path(config.get('etl','landing_zone_path'))

path_target_landing_zone = path_landing_zone / f"produto_interno_bruto_municipios/"
path_target_landing_zone.mkdir(parents=True,exist_ok=True)

#Range dates
year_start  = curent_year = datetime.now().year -2
year_end    = curent_year = datetime.now().year -1


In [0]:
#request data from sidra ibge
for year in range(year_start, year_end+1):
    
    api_url = f"https://apisidra.ibge.gov.br/values/t/5938/n6/all/v/37/p/{year}/d/v37%203"

    response = requests.get(api_url)
    response.raise_for_status()
    data =  response.json()

    path_file = path_target_landing_zone / f"{year}.json"

    with open(path_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)



In [0]:
#define collumns to be used
new_columns = [
      "municipio_Codigo"          #D1C 
    , "municipio"                 #D1N
    , "variavel_Codigo"           #D2C 
    , "variavel"                  #D2N
    , "ano_Codigo"                #D3C
    , "ano"                       #D3N
    , "unidade_de_Medida_Codigo"  #MC          
    , "unidade_de_Medida"         #MN      
    , "nivel_Territorial_Codigo"  #NC           
    , "nivel_Territorial"         #NN      
    , "valor"                     #V  
]

#read json files
df = (
    spark.read
         .option("multiline", "true")
         .option("header", "true")
         .json(str(path_target_landing_zone))
         .toDF(*new_columns) 
)

In [0]:
#/* ------------------------------------------------------------------------------------------------------------------------------------ */
#-- create bronze table
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable("indicadores_brasil.bronze.produto_interno_bruto_municipios")
)

In [0]:
%sql
/* ------------------------------------------------------------------------------------------------------------------------------------ */
-- create silver table
CREATE OR REPLACE TABLE indicadores_brasil.silver.produto_interno_bruto_municipios
select
      municipio_codigo
    , trim(split(municipio, ' - ')[0]) AS municipio_nome
    , trim(split(municipio, ' - ')[1]) AS uf_nome    
    , variavel_codigo
    , variavel
    , ano_codigo
    , ano
    , unidade_de_medida_codigo
    , unidade_de_medida
    , nivel_territorial_codigo
    , nivel_territorial
    , cast(valor as decimal(38,3))
from 
    indicadores_brasil.bronze.produto_interno_bruto_municipios
where 
    lower(municipio_Codigo) <> lower('Município (Código)')

In [0]:
df = spark.sql("""
    select 
        * 
    from 
        indicadores_brasil.silver.produto_interno_bruto_municipios
""")

df = df.toPandas()

df.to_excel(path_project / "data/reports/produto_interno_bruto_municipios.xlsx",index = False)